# 06: Avaliação operacional

Fecha a fase 7 com os cenários D1–D3 do estudo de melhorias. Uma única divisão temporal (notebook 05) produz um único
número por modelo. Aqui:

1. **Várias origens no tempo (D1):** em cada data de corte, o modelo treina com todo o passado e é testado nos
   6 meses seguintes. É o que acontece em produção, repetido 7 vezes.
2. **Métrica de decisão (D2):** quantas faltas o serviço alcança alertando 5%, 10% ou 20% dos agendamentos.
3. **Subgrupos (D3):** para quem o modelo funciona e para quem não funciona.

**Configurações comparadas:**

| Configuração | Variáveis | ID | Ajuste de nível |
|---|---|---|---|
| Linha de base | v1 (25, notebook 02) | exato | não |
| Fase 7 | v7 (38: + histórico recente, suavizado, partida a frio) | aproximado | não |
| Fase 7 + ajuste | v7 | aproximado | sim (taxa dos últimos 6 meses) |

Hiperparâmetros da regressão logística e do LightGBM: escolhidos em cada origem, na validação interna temporal
(últimos 25% do passado), com 15 tentativas do Optuna. `periodo` fica de fora, porque o futuro nunca tem um valor
visto no treino. Requisitos: RF-060 a RF-069 e RNF-005 (`REQUISITOS.md`).

> **Histórico exploratório (fases 7 e 8).** Este notebook foi produzido com o protocolo anterior, **antes da
> correção da sobreposição entre treino e teste** (PR #36) e sem a seleção interna de faixas, ESS e balanceamento.
> Os números e conclusões abaixo registram o processo de desenvolvimento, mas **não sustentam os resultados do
> artigo**. O protocolo final está nos notebooks `03_rede_bayesiana`, `04_experimentos` e
> `08_validacao_sensibilidade_e_base_externa`, com resultados em `resultados/artigo_sem_sobreposicao/`.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from noshow.features import CATEGORICAL, CATEGORICAL_V3, CONTINUOUS, CONTINUOUS_V3
from noshow.operational import auc_by_origin, decision_by_origin, decision_table, run_origins
from noshow.registry import MODELOS, load_bases

warnings.filterwarnings("ignore")
assert os.environ.get("PYTHONHASHSEED") == "0", "Use o kernel do projeto (PYTHONHASHSEED=0); ver README."
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAIDA = RAIZ / "resultados" / "avaliacao_operacional"
RECALCULAR = False  # True refaz as 7 origens (~30–40 min)

ORIGENS = ["2017-07-01", "2018-01-01", "2018-07-01", "2019-01-01", "2019-07-01", "2021-01-01", "2021-07-01"]
CONFIGS = {
    "Linha de base": {"base": "exato", "categoricas": CATEGORICAL, "continuas": CONTINUOUS},
    "Fase 7": {"base": "aproximado", "categoricas": CATEGORICAL_V3, "continuas": CONTINUOUS_V3},
    "Fase 7 + ajuste": {"base": "aproximado", "categoricas": CATEGORICAL_V3, "continuas": CONTINUOUS_V3,
                        "ajuste_nivel": {"meses": 6, "minimo": 500}},
}
bases = load_bases()
if RECALCULAR or not (SAIDA / "predicoes.parquet").exists():
    M, P = run_origins(bases, CONFIGS, ORIGENS)
    SAIDA.mkdir(parents=True, exist_ok=True)
    M.to_csv(SAIDA / "metricas.csv", index=False)
    P.to_parquet(SAIDA / "predicoes.parquet", index=False)
else:
    M, P = pd.read_csv(SAIDA / "metricas.csv"), pd.read_parquet(SAIDA / "predicoes.parquet")

BLUE, ORANGE, GRAY, INK, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#52514e", "#1baf7a"
COR_CFG = {"Linha de base": GRAY, "Fase 7": BLUE, "Fase 7 + ajuste": ORANGE}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})
M.groupby("origem")[["n_treino", "n_teste", "taxa_teste"]].first().style.format(
    {"n_treino": "{:,}", "n_teste": "{:,}", "taxa_teste": "{:.1%}"})

## 1. Desempenho em cada origem (D1)

In [ ]:
fig, axes = plt.subplots(1, len(MODELOS), figsize=(16, 3.8), sharey=True)
for ax, m in zip(axes, MODELOS):
    for cfg, cor in COR_CFG.items():
        s = M[(M.modelo == m) & (M.config == cfg)]
        ax.plot(s.origem.astype(str), s.roc_auc, "o-", color=cor, lw=2, ms=6, mec="white", mew=1.5, label=cfg)
    ax.axhline(0.5, color=GRAY, ls="--", lw=1)
    ax.set_title(m); ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0].set_ylabel("ROC-AUC"); axes[-1].legend(frameon=False, fontsize=8)
fig.suptitle("ROC-AUC nos 6 meses seguintes a cada origem", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

resumo = M.groupby(["modelo", "config"]).agg(roc_media=("roc_auc", "mean"), roc_dp=("roc_auc", "std"),
                                              pr_media=("pr_auc", "mean"), brier=("brier", "mean"))
resumo.loc[[(m, c) for m in MODELOS for c in CONFIGS]].style.format("{:.4f}") \
    .highlight_max(subset=["roc_media", "pr_media"], color="#d6e6f8")

**Comparações pareadas por origem** (Wilcoxon, 7 pares; o menor *p* bilateral possível é 0,016).

In [ ]:
linhas = []
for m in MODELOS:
    piv = M[M.modelo == m].pivot_table(index="origem", columns="config", values="roc_auc")
    for a, b in [("Fase 7", "Linha de base"), ("Fase 7 + ajuste", "Fase 7")]:
        d = piv[a] - piv[b]
        linhas.append({"modelo": m, "comparação": f"{a} × {b}", "diferença_média": d.mean(),
                       "origens_a_maior": int((d > 0).sum()), "p": stats.wilcoxon(piv[a], piv[b]).pvalue})
pd.DataFrame(linhas).set_index(["modelo", "comparação"]).style.format(
    {"diferença_média": "{:+.4f}", "p": "{:.3f}"}).map(lambda v: "font-weight: bold" if v < 0.05 else "", subset=["p"])

## 2. Métrica de decisão (D2)

Em produção, o serviço alerta os pacientes de maior risco **dentro do período que está prevendo**. Por isso, a tabela
é calculada **em cada origem** (alertando os 5%, 10% ou 20% de maior risco daqueles 6 meses) e depois é feita a média
ponderada pelo tamanho do teste. Juntar as predições de todas as origens e ordená-las juntas premiaria quem acerta o
nível de risco *entre* períodos, uma comparação que o serviço nunca faz.

In [ ]:
melhor = M.groupby(["config", "modelo"]).roc_auc.mean().groupby("config").idxmax().map(lambda t: t[1])
tabelas = []
for cfg, m in melhor.items():
    for origem, s in P[(P.config == cfg) & (P.modelo == m)].groupby("origem"):
        tabelas.append(decision_table(s.y.to_numpy(), s.p.to_numpy()).assign(config=cfg, modelo=m, n=len(s)))
t = pd.concat(tabelas)
cols = ["alertas_por_1000", "recall", "precisao", "ganho", "faltas_alcancadas_por_1000", "faltas_ao_acaso_por_1000"]
dec = t.groupby(["config", "modelo", "volume"]).apply(lambda g: pd.Series(np.average(g[cols], axis=0, weights=g.n), cols))
dec.style.format({"alertas_por_1000": "{:.0f}", "recall": "{:.1%}", "precisao": "{:.1%}", "ganho": "{:.2f}×",
                  "faltas_alcancadas_por_1000": "{:.1f}", "faltas_ao_acaso_por_1000": "{:.1f}"})

## 3. Subgrupos (D3)

Desempenho da melhor configuração por grupo de pacientes, calculado em cada origem e combinado pela média ponderada
pelo tamanho do grupo (pelo mesmo motivo da seção 2). Em cada origem, grupos com menos de 100 agendamentos ou sem as
duas classes ficam de fora.

In [ ]:
cfg_final = M.groupby("config").roc_auc.mean().idxmax()
m_final = melhor[cfg_final]
s = P[(P.config == cfg_final) & (P.modelo == m_final)].reset_index(drop=True)
info = bases["aproximado"].iloc[s.linha.to_numpy()].reset_index(drop=True)
hist = pd.cut(info.n_agendamentos_anteriores, [-1, 0, 4, 9, 29, 10_000],
              labels=["0 (novo)", "1–4", "5–9", "10–29", "30+"]).astype("string").fillna("sem ID")
print(f"Configuração: {cfg_final} · modelo: {m_final} · {len(s):,} agendamentos previstos")
grupos = {"faixa_etaria": info.faixa_etaria, "specialty_grp": info.specialty_grp, "histórico": hist,
          "id_qualidade": info.id_qualidade}

pd.concat({nome: auc_by_origin(s, g) for nome, g in grupos.items()}).style.format(
    {"n": "{:,.0f}", "taxa": "{:.1%}", "roc_auc": "{:.3f}", "origens": "{:.0f}"}, na_rep="—") \
    .background_gradient(subset=["roc_auc"], cmap="RdBu", vmin=0.4, vmax=0.7)

## 4. Cenário com ID: aproximação do uso em produção

Não há acesso à fonte da base, então o ID real e a data de marcação não podem ser obtidos. Em produção, todo paciente
tem identificador e constrói histórico desde a primeira consulta. Os registros desta base que **têm** ID são a melhor
aproximação desse cenário.

> **Ressalva:** o recorte é uma aproximação, não uma simulação. Os registros sem ID se concentram em 2021–2022 e
> podem ter perfil diferente dos demais.

In [ ]:
qualidade = bases["aproximado"].id_qualidade
cenarios = {"base completa": None, "com ID (exato)": ["exato"], "com ID (exato + aproximado)": ["exato", "aproximado"]}
linhas = []
for cfg, m in [(cfg_final, m_final), ("Linha de base", melhor["Linha de base"])]:
    s_ = P[(P.config == cfg) & (P.modelo == m)].reset_index(drop=True)
    q = qualidade.iloc[s_.linha.to_numpy()].to_numpy()
    for nome, niveis in cenarios.items():
        r = s_ if niveis is None else s_[np.isin(q, niveis)]
        dec10 = decision_by_origin(r, volumes=(0.10,)).loc[0.10]
        linhas.append({"configuração": f"{cfg} ({m})", "cenário": nome, "agendamentos": len(r),
                       "roc_auc": auc_by_origin(r).loc["todos", "roc_auc"],
                       "faltas_por_1000_alertando_10%": dec10.faltas_alcancadas_por_1000,
                       "ao_acaso": dec10.faltas_ao_acaso_por_1000, "ganho": dec10.ganho})
pd.DataFrame(linhas).set_index(["configuração", "cenário"]).style.format(
    {"agendamentos": "{:,}", "roc_auc": "{:.3f}", "faltas_por_1000_alertando_10%": "{:.1f}", "ao_acaso": "{:.1f}",
     "ganho": "{:.2f}×"})

## 5. Retroalimentação: o modelo melhora à medida que o paciente acumula histórico?

Premissa do projeto: pacientes sem histórico o constroem ao longo do uso, e o modelo fica mais assertivo. O mecanismo
já existe em dois níveis: o histórico é recalculado a cada desfecho (RF-013) e o modelo é retreinado periodicamente
(RF-046). Aqui ele é **medido**.

In [ ]:
faixas = [-1, 0, 1, 2, 4, 9, 19, 29, 49, 10_000]
rotulos = ["0", "1", "2", "3–4", "5–9", "10–19", "20–29", "30–49", "50+"]
hist_fino = pd.cut(info.n_agendamentos_anteriores, faixas, labels=rotulos).astype("string").fillna("sem ID")
curva = auc_by_origin(s, hist_fino, minimo=80).reindex(rotulos + ["sem ID"])

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(rotulos, curva.loc[rotulos, "roc_auc"], "o-", color=BLUE, lw=2, ms=7, mec="white", mew=1.5)
ax.axhline(0.5, color=GRAY, ls="--", lw=1)
ax.axhline(curva.loc["sem ID", "roc_auc"], color=ORANGE, ls=":", lw=1.5)
ax.text(len(rotulos) - 1, curva.loc["sem ID", "roc_auc"], "  sem ID", color=ORANGE, va="center", fontsize=8)
ax.set_xlabel("consultas anteriores do paciente"); ax.set_ylabel("ROC-AUC (dentro de cada origem)")
ax.set_title("Curva de aprendizado por paciente")
plt.tight_layout(); plt.show()
curva.style.format({"n": "{:,.0f}", "taxa": "{:.1%}", "roc_auc": "{:.3f}", "origens": "{:.0f}"}, na_rep="—")

In [ ]:
conhecidos = bases["aproximado"].dropna(subset=["patient_id"])
dias_paciente = conhecidos.drop_duplicates(["patient_id", "appointment_date"]).sort_values("appointment_date") \
    .groupby("patient_id").appointment_date.agg(list)
ate_5 = dias_paciente[dias_paciente.str.len() >= 5].map(lambda d: (d[4] - d[0]).days)
print(f"Tempo até o paciente acumular 5 dias de consulta: mediana {ate_5.median():.0f} dias "
      f"(P25 {ate_5.quantile(.25):.0f}, P75 {ate_5.quantile(.75):.0f}); {len(ate_5):,} pacientes")

por_origem = M[(M.config == cfg_final)].pivot_table(index="origem", columns="modelo", values="roc_auc")
por_origem["treino"] = M.groupby("origem").n_treino.first()
rho = stats.spearmanr(por_origem.treino, por_origem[m_final]).statistic
print(f"Volume de treino × ROC-AUC ao longo das origens ({m_final}): correlação de Spearman {rho:+.2f}")
por_origem.style.format({**{m: "{:.3f}" for m in MODELOS}, "treino": "{:,}"})

**Recomendação para produção.** Quando a mensageria passar a agir sobre as predições, um paciente lembrado que
comparece entra no histórico como "não faltou", e o modelo aprenderia que aquele perfil é de baixo risco, quando foi o
lembrete que funcionou. Para a retroalimentação continuar válida, o sistema deve **registrar cada intervenção** (quem
foi lembrado e quando) separada do desfecho, e manter um **grupo de controle**: uma pequena fração de pacientes de
alto risco sem lembrete, para medir o efeito real da intervenção.

## 6. Síntese

### Resultado

| Comparação (ROC-AUC média em 7 origens) | Linha de base | Fase 7 | Diferença | Origens melhores |
|---|---|---|---|---|
| **LightGBM** | 0,534 | **0,549** | **+0,016 (*p* = 0,016)** | **7 de 7** |
| Regressão logística | 0,542 | **0,558** | +0,016 (*p* = 0,078) | 5 de 7 |
| RB (AIC) | 0,510 | 0,524 | +0,014 (n.s.) | 4 de 7 |
| RB (BIC) | 0,515 | 0,510 | −0,005 (n.s.) | 1 de 7 |

- **As variáveis da fase 7 melhoram de forma consistente.** Com o LightGBM, o ganho aparece nas 7 origens, o
  resultado mais robusto de todos os experimentos.
- **O ajuste de nível não muda nada aqui:** métricas idênticas com e sem ajuste, porque a correção é monotônica e a
  avaliação é feita dentro de cada período.
- **Prevendo os 6 meses seguintes, a discriminação real é fraca:** ROC-AUC de 0,51 a 0,56. As AUCs maiores da divisão
  temporal única (~0,57) incluíam o efeito do nível de risco *entre* períodos.

### O que isso vale para o serviço (melhor modelo: regressão logística, fase 7)

| Alertando os de maior risco | Faltas alcançadas por 1.000 agendamentos | Sorteando | Ganho |
|---|---|---|---|
| 5% | 9,2 | 5,4 | 1,75× |
| 10% | 15,6 | 10,8 | 1,52× |
| 20% | 27,4 | 21,5 | 1,33× |

Comparada à linha de base, a fase 7 alcança cerca de **2 faltas a mais por 1.000 agendamentos** com 10% de alertas
(15,6 contra 13,7), com o mesmo volume de contatos.

### Para quem o modelo não funciona

- **Pacientes novos** (AUC 0,497) e **sem ID** (0,485) ficam no nível do acaso. Sem histórico, não há sinal
  suficiente, e as variáveis de partida a frio não resolveram.
- **Idosos (60+)** (0,515) e **adolescentes** (0,534) ficam perto do acaso; a primeira infância (0–5, 0,581) e a
  fonoaudiologia (0,583) são os grupos mais previsíveis.
- Isso deve entrar no TCC como **limitação de equidade**: o modelo ajuda mais quem já tem histórico no serviço.

### Cenário com ID (proxy de produção)

| Regressão logística, fase 7 | ROC-AUC | Alertando 10% dos agendamentos |
|---|---|---|
| Base completa | 0,556 | 15,6 faltas por 1.000 (**1,52×** o acaso) |
| **Só pacientes com ID exato** | **0,568** | **16,2** faltas por 1.000 (**1,72×** o acaso) |

Onde o paciente tem identificador, e portanto constrói histórico, o modelo rende mais. Em produção, todo paciente tem
ID, então este é o patamar esperado de uso, com a ressalva de que o recorte é uma aproximação.

### Retroalimentação

- **O paciente que constrói histórico fica mais previsível:** de 0,490 (nenhuma consulta anterior) para 0,573 entre
  a 5ª e a 9ª consulta. A mediana até o 5º dia de consulta é de **35 dias** de terapia.
- **Depois disso, o ganho não continua:** com 20 ou mais consultas, a AUC volta para cerca de 0,52. Para pacientes
  antigos, a taxa de faltas acumulada deixa de diferenciar quem vai faltar.
- **Mais dados no modelo não aumentam a assertividade:** ao longo das 7 origens, o treino cresce de 15 mil para 44 mil
  agendamentos sem tendência de melhora (correlação de −0,43). Retreinar serve para acompanhar a deriva, não para
  acumular precisão.
- **A premissa se confirma no início do relacionamento com o paciente** e depende de ele ter ID. Os registros sem ID
  ficam no nível do acaso (0,485), e a reconstrução do ID foi esgotada (notebook 07).
- **Em produção,** é preciso registrar as intervenções e manter um grupo de controle, para que a retroalimentação não
  aprenda o efeito do próprio lembrete.

### Conclusão das fases 7 e 8

| Cenário do estudo | Resultado |
|---|---|
| C1/C2: ajuste do nível de risco (v6) | **Não ajudou**: instável com a taxa mensal volátil; irrelevante para alertas por volume |
| B1–B3: histórico suavizado, recência, partida a frio (v7) | **Ajudou pouco, mas de forma consistente**: +0,016 em 7/7 origens; melhor agrupada 0,690 |
| B4: rótulo limpo (v8) | **Não ajudou**: ruído pequeno demais para mudar o modelo |
| D1–D3: avaliação operacional (06) | Feita; mostra ganho real de 1,5× sobre o acaso com 10% de alertas e os grupos onde o modelo falha |

O teto continua dado pela informação disponível. O próximo salto depende dos cenários A1/A2 do estudo: a data de
marcação e um ID real de paciente.